In [ ]:
!pip install -q pandas umls-python-client #

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
add_snomed_to_problem_list.py
-----------------------------
For each patient in the input JSON, resolve every problem_list entry to a SNOMED CT
concept via the UMLS API, add a new key `problem_list_snomed`, and save to a new file.

Pipeline per problem:  free text  --UMLS search (SNOMEDCT_US)-->  SNOMED CT code + name

Requires the `umls-python-client` package and a UMLS API key (free with a UMLS
license at https://uts.nlm.nih.gov/uts/).  Install:  pip install umls-python-client
"""
import json
from umls_python_client import UMLSClient

# ============================ CONFIG ============================
UMLS_API_KEY = "TBD"          # <-- put your key here
INPUT_FILE   = "/content/pmr_synth_parsed_sample.json"          # input patient JSON
OUTPUT_FILE  = "/content/pmr_synth_parsed_sample_snomed.json"   # output with SNOMED added
# ===============================================================


def _results(resp):
    """Pull the results list out of a UMLS response (may be a dict or JSON string)."""
    if isinstance(resp, str):
        try:
            resp = json.loads(resp)
        except json.JSONDecodeError:
            return []
    if isinstance(resp, list):
        return resp
    r = resp.get("result", resp)
    return r.get("results", []) if isinstance(r, dict) else (r or [])


In [ ]:
def _search_cui(client, text, search_type):
    """One search attempt -> (cui, name) or (None, None)."""
    resp = client.searchAPI.search(
        text, return_id_type="concept", search_type=search_type,
        page_size=1, return_indented=False, format="json")
    res = _results(resp)
    if res and res[0].get("ui") not in (None, "NONE", "None"):
        return res[0]["ui"], res[0].get("name")
    return None, None


In [ ]:
def _simplify(text):
    """Reduce a verbose ICD/billing description to its clinical core.
    e.g. 'Class 3 severe obesity with body mass index (BMI) of 45.0 to 49.9 in
    adult, unspecified ...'  ->  'severe obesity'."""
    import re
    s = text.split(",")[0]                                  # drop trailing ICD clauses
    s = re.sub(r"\([^)]*\)", " ", s)                        # remove parentheticals (BMI), (CMC)
    s = re.split(r"\bwith\b", s, maxsplit=1, flags=re.I)[0] # cut at " with <measurement/qualifier>"
    s = re.sub(r"^\s*(class|type|grade|stage)\s+\S+\s+", "", s, flags=re.I)  # drop 'Class 3 ' prefix
    return re.sub(r"\s+", " ", s).strip()




In [ ]:
def _query_variants(text):
    """Ordered, de-duplicated queries to try: most specific first."""
    variants = [text]
    core = text.split(",")[0].strip()
    if core and core.lower() != text.lower():
        variants.append(core)
    simplified = _simplify(text)
    if simplified and simplified.lower() not in (v.lower() for v in variants):
        variants.append(simplified)
    return variants

In [ ]:
def resolve_to_snomed(client, text):
    """Free text -> UMLS concept (CUI) -> SNOMED CT code + name.

    Searches ALL vocabularies (not just SNOMED) so ICD-style phrasing matches, then
    crosswalks the CUI to its SNOMED CT atom. Tries progressively simpler queries
    (full string -> text before first comma -> cleaned clinical core) with VALID
    search types only ('words', 'normalizedString'; NOTE 'approximate' is NOT valid
    for UMLS /search). Returns a null record if nothing resolves.

    NOTE: verbose ICD/billing diagnosis rubrics resolve less reliably than clinician
    problem-list terms; the problem_list is the source intended for SNOMED encoding.
    """
    null = {"input_text": text, "cui": None, "sctid": None, "snomed_name": None}

    cui = cui_name = None
    for query in _query_variants(text):
        for stype in ("words", "normalizedString"):
            try:
                cui, cui_name = _search_cui(client, query, stype)
            except Exception as e:                          # try next variant, don't abort
                print(f"    ! search failed ({stype}) for {query!r}: {e}")
                continue
            if cui:
                break
        if cui:
            break
    if not cui:
        return null

    # Crosswalk the concept to SNOMED CT by pulling its SNOMEDCT_US atom(s).
    try:
        atoms = _results(client.cuiAPI.get_atoms(
            cui, sabs="SNOMEDCT_US", page_size=25, return_indented=False, format="json"))
    except Exception as e:
        print(f"    ! get_atoms failed for {cui} ({text!r}): {e}")
        return {**null, "cui": cui}
    for atom in atoms:
        code = (atom.get("code") or "").rstrip("/").split("/")[-1]
        if code.isdigit():
            return {"input_text": text, "cui": cui,
                    "sctid": code, "snomed_name": atom.get("name") or cui_name}
    return {**null, "cui": cui}   # concept found, but it has no SNOMED CT atom

In [ ]:
def main():
    client = UMLSClient(api_key=UMLS_API_KEY)

    with open(INPUT_FILE) as f:
        patients = json.load(f)

    cache = {}   # remember terms we've already looked up (problem lists repeat a lot)
    for patient in patients:
        problem_snomed = []
        for problem in patient.get("problem_list", []):
            if problem not in cache:
                cache[problem] = resolve_to_snomed(client, problem)
            problem_snomed.append(cache[problem])
        patient["problem_list_snomed"] = problem_snomed          # append the new key
        n_ok = sum(1 for p in problem_snomed if p["sctid"])
        print(f"{patient.get('id', '?')}: resolved {n_ok}/{len(problem_snomed)} problems")

        diagnosis_snomed = []
        for diagnosis in patient.get("diagnoses", {}).get("past_year", []):
            if diagnosis not in cache:
                cache[diagnosis] = resolve_to_snomed(client, diagnosis)
            diagnosis_snomed.append(cache[diagnosis])
        patient["diagnosis_list_snomed"] = diagnosis_snomed          # append the new key
        n_ok = sum(1 for p in diagnosis_snomed if p["sctid"])
        print(f"{patient.get('id', '?')}: resolved {n_ok}/{len(diagnosis_snomed)} diagnosis")

    with open(OUTPUT_FILE, "w") as f:
        json.dump(patients, f, indent=2)
    print(f"\nWrote {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


INBOX_A-00: resolved 10/20 problems
INBOX_A-00: resolved 5/7 diagnosis
INBOX_A-01: resolved 6/10 problems
INBOX_A-01: resolved 9/10 diagnosis
INBOX_A-02: resolved 4/6 problems
INBOX_A-02: resolved 3/5 diagnosis

Wrote /content/pmr_synth_parsed_sample_snomed.json


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ckm_stage_estimator.py
--------------------
CKM Stage 0-4 from a list of SNOMED CT codes, utilizing a Local Mock Ontology
for offline hierarchical (is-a) subsumption.
"""
import json

# --------------------------------------------------------------------------- #
# Tier parent concepts (SNOMED CT). The "Pure" Clinical Guidelines
# --------------------------------------------------------------------------- #

# Clinical CVD (Table 4: CHD, HF, stroke, PAD, AF) -> Stage 4 (with overlap)
CLINICAL_CVD = [
    "84114007",   # Heart failure                 (VERIFIED)
    "22298006",   # Myocardial infarction         (VERIFIED)
    "53741008",   # Coronary arteriosclerosis/CHD (VERIFIED)
    "49436004",   # Atrial fibrillation           (VERIFIED)
    "230690007",  # Cerebrovascular accident      (VERIFIED)
    "40275004",   # Peripheral vascular disease   (VERIFIED)
    "85898001",   # Cardiomyopathy (general)      (VERIFIED)
    "60234000",   # Aortic valve insufficiency    (VERIFIED)
]

# Kidney failure / very-high-risk CKD -> Stage 3 alone, or 4b modifier with clinical CVD
VERY_HIGH_CKD = [
    "46177005",   # End-stage renal disease       (VERIFIED)
]

# Metabolic risk factors and moderate-high-risk CKD -> Stage 2
STAGE_2_METABOLIC_CKD = [
    "44054006",   # Type 2 diabetes mellitus      (VERIFIED)
    "73211009",   # Diabetes mellitus             (VERIFIED)
    "38341003",   # Hypertensive disorder         (VERIFIED)
    "59621000",   # Essential hypertension        (VERIFIED)
    "55822004",   # Hyperlipidemia                (VERIFIED)
    "13644009",   # Hypercholesterolemia          (VERIFIED)
    "709044004",  # Chronic kidney disease        (VERIFIED)
    "90688005",   # Chronic renal failure         (VERIFIED)
    "34436003",   # Albuminuria                   (VERIFIED)
]

# Adiposity -> Stage 1
STAGE_1_ADIPOSITY = [
    "414916001",  # Obesity (Parent concept)      (VERIFIED)
    "238131007",  # Overweight                    (VERIFIED)
    "15777000",   # Prediabetes                   (VERIFIED)
]

# --------------------------------------------------------------------------- #
# LOCAL MOCK ONTOLOGY (METHOD 1)
# --------------------------------------------------------------------------- #
class LocalMockOntology:
    """
    A local offline dictionary to simulate SNOMED CT hierarchical relationships.
    Map specific patient codes (children) to the Stage bucket codes (parents).
    """
    def __init__(self):
        # Format: "Patient's Exact Child Code": {"Parent Bucket Code"}
        self.ancestors = {

            # --- STAGE 4 MAPPINGS (Clinical CVD) ---
            "195108009": {"84114007"},  # Congestive heart failure -> Heart failure
            "8957000": {"53741008"},    # Coronary artery disease NOS -> Coronary arteriosclerosis
            "155364009": {"49436004"},  # Atrial fibrillation, unspecified type -> Atrial fibrillation
            "195196001": {"230690007"}, # Transient ischemic attack -> Cerebrovascular accident
            "441530006": {"85898001"},  # Takotsubo cardiomyopathy -> Cardiomyopathy (general)
            "195081002": {"49436004"},  # Map Paroxysmal -> Atrial fibrillation

            # --- STAGE 3 MAPPINGS (Very High-Risk CKD) ---
            # (Reserved space for specific CKD Stage 4/5 codes to map to ESRD/Renal Failure)

            # --- STAGE 2 MAPPINGS (Metabolic Risk & CKD) ---
            "194757006": {"59621000"},  # Essential hypertension (specific) -> Essential hypertension (parent)
            "154739000": {"55822004"},  # Hyperlipidemia (unspecified) -> Hyperlipidemia (parent)
            "267500001": {"13644009"},  # Hypercholesterolemia (specific) -> parent
            "267434003": {"55822004"},  # Mixed hyperlipidemia -> Hyperlipidemia parent

            # --- STAGE 1 MAPPINGS (Adiposity & Impaired Glycemia) ---
            "5476005": {"414916001"},   # Obesity (general) -> Obesity (parent)
            "389986000": {"414916001"}, # Morbid obesity -> Obesity (parent)
        }

    def is_a(self, code, parent_code):
        # 1. Reflexive check: If the code is already an exact match to the parent list
        if str(code) == str(parent_code):
            return True
        # 2. Hierarchical check: If the code maps to a parent in our dictionary
        return str(parent_code) in self.ancestors.get(str(code), set())


# --------------------------------------------------------------------------- #
# EVALUATION LOGIC
# --------------------------------------------------------------------------- #
def approx_ckm_stage(entries, ontology):
    """Return (stage, reasons) for a list of SNOMED mapping dicts using ontology."""

    def matches(parent_list, name):
        matched = []
        for e in entries:
            if isinstance(e, dict) and e.get("sctid"):
                patient_code = str(e["sctid"])
                # Check this patient code against every parent in the target tier
                for parent in parent_list:
                    if ontology.is_a(patient_code, parent):
                        matched.append({
                            "table": name,
                            "sctid": patient_code,
                            "matched_parent": parent, # Shows which bucket code it triggered
                            "snomed_name": e.get("snomed_name", "Unknown")
                        })
                        break # Once it matches one parent in the list, stop checking the list
        hits[name] = matched
        return bool(hits[name])

    hits = {}
    has_cvd            = matches(CLINICAL_CVD, "CLINICAL_CVD")
    has_kidney_failure = matches(VERY_HIGH_CKD, "VERY_HIGH_CKD")
    has_metabolic      = matches(STAGE_2_METABOLIC_CKD, "STAGE_2_METABOLIC_CKD")
    has_adiposity      = matches(STAGE_1_ADIPOSITY, "STAGE_1_ADIPOSITY")

    if has_cvd:
        stage, used = 0, ["CLINICAL_CVD"]          # isolated CVD -> not CKM
    elif has_kidney_failure:
        stage, used = 3, ["VERY_HIGH_CKD"]
    elif has_metabolic:
        stage, used = 2, ["STAGE_2_METABOLIC_CKD"]
    elif has_adiposity:
        stage, used = 1, ["STAGE_1_ADIPOSITY"]
    else:
        stage, used = 0, []

    return stage, [h for n in used for h in hits[n]]

# --------------------------------------------------------------------------- #
# JSON Processing Execution Block
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    input_file = "ckm_pts_segments_1.json"
    output_file = "ckm_pts_segments_1_staged.json"

    print(f"Reading patient data from {input_file}...")

    # Instantiate the local dictionary "ontology"
    local_ontology = LocalMockOntology()

    try:
        with open(input_file, 'r') as f:
            patients = json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find {input_file}. Please ensure it is in the same directory.")
        exit(1)

    for patient in patients:
        patient_id = patient.get("id", "Unknown_ID")
        unique_entries = {}

        # Extract SNOMED codes from the problem list
        for problem in patient.get("problem_list_snomed", []):
            sctid = problem.get("sctid")
            if sctid is not None:
                unique_entries[str(sctid)] = problem

        # Extract SNOMED codes from the diagnosis list
        for diagnosis in patient.get("diagnosis_list_snomed", []):
            sctid = diagnosis.get("sctid")
            if sctid is not None:
                unique_entries[str(sctid)] = diagnosis

        entries_list = list(unique_entries.values())

        # Pass the ontology object into the evaluator
        stage_result, reasons = approx_ckm_stage(entries_list, local_ontology)

        patient["ckm_stage"] = stage_result
        patient["ckm_reasons"] = reasons

        print(f"Processed {patient_id}: Found {len(entries_list)} unique SNOMED codes -> Assigned CKM Stage {stage_result}")

    print(f"\nSaving updated patient data to {output_file}...")
    with open(output_file, 'w') as f:
        json.dump(patients, f, indent=2)

    print("Processing complete.")

Reading patient data from ckm_pts_segments_1.json...
Processed INBOX_A-00000-of-00001-00: Found 14 unique SNOMED codes -> Assigned CKM Stage 2
Processed INBOX_A-00000-of-00001-01: Found 14 unique SNOMED codes -> Assigned CKM Stage 0
Processed INBOX_A-00000-of-00001-02: Found 7 unique SNOMED codes -> Assigned CKM Stage 1

Saving updated patient data to ckm_pts_segments_1_staged.json...
Processing complete.


In [ ]:
## Debug code below

In [20]:
import json
import random
from datetime import datetime, timedelta

def get_random_2025_datetime():
    start = datetime(2025, 1, 1)
    end = datetime(2025, 12, 31, 23, 59, 59)
    random_date = start + timedelta(
        seconds=random.randint(0, int((end - start).total_seconds()))
    )
    return random_date.strftime("%Y-%m-%dT%H:%M:%S")

def convert_to_staged_format(input_filepath, output_filepath):
    # 1. Load the original JSON array
    with open(input_filepath, 'r') as infile:
        raw_patients = json.load(infile)

    # 2. Define the root-level metadata
    staged_data = {
        "schema_version": "1.0",
        "corpus_id": "ckm_ambiguity_probe_v1",
        "notice": "SYNTHETIC DATA. All patients, encounters, laboratory values, and messages are fabricated for model evaluation. No real patient data. Not for clinical use.",
        "description": "Simulated cardiovascular-kidney-metabolic (CKM) patients with structured EHR context and ambiguous patient-portal messages. Each message carries an expected triage label for automated scoring. Several messages are near-duplicates in free text but differ in expected triage because the structured EHR data disambiguates them - these are flagged with requires_ehr=true and are the core probe items.",
        "triage_levels": [
            {
                "code": "EMERGENT",
                "rank": 3,
                "definition": "Patient has life-threatening issue that needs immediate attention."
            },
            {
                "code": "URGENT",
                "rank": 2,
                "definition": "Patient has non life-threatening issue that would benefit from same-day treatment (e.g. urgent care)."
            },
            {
                "code": "ROUTINE",
                "rank": 1,
                "definition": "Patient should see their doctor sometime in the near future (could be more than 3 days), or patient should make an appointment with a doctor soon (1-3 days)."
            },
            {
                "code": "SELF_CARE",
                "rank": 0,
                "definition": "Patient presents something that is a non-issue and no further steps are needed, or patient has symptoms that can be treated at home, and would benefit from a message instructing them on what to do"
            }
        ],
        "ambiguity_levels": [
            "low",
            "moderate",
            "high"
        ],
        "patients": []
    }

    # Helper mapping for triage levels based on the provided sample
    triage_map = {
        1: "EMERGENT",
        2: "URGENT",
        4: "ROUTINE"
    }

    # 3. Restructure each patient record
    for index, patient in enumerate(raw_patients):
        # Extract and remove the properties moving to the nested 'messages' list
        message_text = patient.pop("patient_message")
        level = patient.pop("level")
        relevancy = patient.pop("relevancy")

        # Generate a random 2025 date/time
        sent_at = get_random_2025_datetime()

        # Build the message object
        message_obj = {
            "message_id": f"{patient['id']}-M1",
            "sent_at": sent_at,
            "text": message_text,
            "expected": {
                "triage": triage_map.get(level, "SELF_CARE"),
                "level": level,
                "relevancy": relevancy
            }
        }

        # Add the nested messages array to the patient
        patient["messages"] = [message_obj]

        # Append to the new root patients array
        staged_data["patients"].append(patient)

    # 4. Write the staged JSON file
    with open(output_filepath, 'w') as outfile:
        json.dump(staged_data, outfile, indent=2)

    print(f"Successfully converted data and saved to {output_filepath}")

# Execute the function
if __name__ == "__main__":
    input_file = "ckm_pts_segments_1.json"
    output_file = "ckm_pts_segments_1_v1.json"

    # Uncomment the line below to run the conversion if your files exist in the same directory:
    convert_to_staged_format(input_file, output_file)

Successfully converted data and saved to ckm_pts_segments_1_v1.json


In [2]:
import json
import os

# Define the input and output filenames
input_filename = "ckm_pts_segments_1_staged_v1.json"

# Construct the output filename by appending "_stg" before the extension
base, ext = os.path.splitext(input_filename)
output_filename = f"{base}_stg{ext}"

print(f"Reading data from: {input_filename}")

try:
    with open(input_filename, 'r') as f:
        full_data = json.load(f)
except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found.")
    full_data = {} # Initialize as empty dict to prevent further errors

# Assuming patient data is under a 'patients' key
patients_list = full_data.get('patients', [])

if patients_list:
    for patient in patients_list:
        # Ensure patient is a dictionary before trying to pop keys
        if isinstance(patient, dict):
            # Remove 'ckm_stage' and 'ckm_reasons' if they exist
            patient.pop('ckm_stage', None)
            patient.pop('ckm_reasons', None)
        else:
            print(f"Warning: Skipping non-dictionary item in patients list: {patient}")

    print(f"Saving modified data to: {output_filename}")
    with open(output_filename, 'w') as f:
        json.dump(full_data, f, indent=2)
    print("Processing complete. Keys 'ckm_stage' and 'ckm_reasons' removed from patient records.")
else:
    print("No patient data to process or input file was empty/not found/did not contain a 'patients' key.")

Reading data from: ckm_pts_segments_1_staged_v1.json
Saving modified data to: ckm_pts_segments_1_staged_v1_stg.json
Processing complete. Keys 'ckm_stage' and 'ckm_reasons' removed from patient records.


In [ ]:
%%writefile parse_pmr_parquet.py
#!/usr/bin/env python3
"""
parse_pmr_parquet.py

Load a PMR-synth .parquet export, parse the packed `text` column into structured
fields, and emit JSON matching the schema of pmr_synth_parsed_sample.json.

Expected input schema
---------------------
    text       : str    -- packed EHR block + patient message
    level      : int64  -- acuity label
    relevancy  : int64  -- EHR-relevancy label

Expected `text` layout
----------------------
    ### EHR: ###Demographics###
    Age: <str>
    Gender: <str>

    ###Full Active Problem List###:
    <item> - <item> - ...

    ###Recent Encounters (Max 10)###

    Diagnoses (Past Year): <item> - <item> - ...
    Diagnoses (Older): <item> - <item> - ...

    ###Medications (Outpatient)###

    Active (Start Date Before Message, Not Yet Ended):
    -<med>
    -<med>

    ### Patient Message: <free text>

Output record
-------------
    {
      "patient_message": str,
      "demographics": {"age": str, "gender": str},
      "problem_list": [str, ...],
      "diagnoses": {"past_year": [str, ...], "older": [str, ...]},
      "medications": [str, ...],
      "level": int | None,
      "relevancy": int | None,
      "id": "<PREFIX>-<NN>"
    }

Usage
-----
    python parse_pmr_parquet.py INBOX_A-one_pt.parquet -o out.json
    python parse_pmr_parquet.py *.parquet -o corpus.json --keep-empty
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

import pandas as pd

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

# The sample JSON contains two parser artifacts (see README notes in the
# accompanying message). Set to True to reproduce them bug-for-bug so new
# records are byte-compatible with an already-parsed corpus; False (default)
# to emit clean data.
LEGACY_QUIRKS = False

# Section markers, in document order.
MSG_MARKER = "### Patient Message:"
SEC_DEMOGRAPHICS = "###Demographics###"
SEC_PROBLEMS = "###Full Active Problem List###"
SEC_ENCOUNTERS = "###Recent Encounters (Max 10)###"
SEC_MEDS = "###Medications (Outpatient)###"

# Items inside list-valued fields are joined with " - ".
LIST_SEP = re.compile(r"\s+-\s+")


# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------

def _split_items(blob: str) -> list[str]:
    """Split a ' - '-delimited run into a clean list of items."""
    blob = (blob or "").strip()
    if not blob:
        return []
    return [p.strip() for p in LIST_SEP.split(blob) if p.strip()]


def _section(text: str, start: str, end: str | None) -> str:
    """Return the text between two markers. Empty string if `start` absent."""
    i = text.find(start)
    if i == -1:
        return ""
    i += len(start)
    if end:
        j = text.find(end, i)
        if j != -1:
            return text[i:j]
    return text[i:]


def _to_int(value) -> int | None:
    """Cast a possibly-null numeric label to int, preserving None."""
    if value is None or pd.isna(value):
        return None
    return int(value)


# --------------------------------------------------------------------------
# Field parsers
# --------------------------------------------------------------------------

def parse_demographics(ehr: str) -> dict[str, str]:
    block = _section(ehr, SEC_DEMOGRAPHICS, SEC_PROBLEMS)
    age = re.search(r"Age:\s*(.*)", block)
    gender = re.search(r"Gender:\s*(.*)", block)
    return {
        "age": age.group(1).strip() if age else "",
        "gender": gender.group(1).strip() if gender else "",
    }


def parse_problem_list(ehr: str) -> list[str]:
    block = _section(ehr, SEC_PROBLEMS, SEC_ENCOUNTERS)
    return _split_items(block.lstrip(":").strip())


def parse_diagnoses(ehr: str) -> dict[str, list[str]]:
    block = _section(ehr, SEC_ENCOUNTERS, SEC_MEDS)

    past = re.search(r"Diagnoses \(Past Year\):(.*)", block)
    older = re.search(r"Diagnoses \(Older\):(.*)", block)

    out = {
        "past_year": _split_items(past.group(1) if past else ""),
        "older": _split_items(older.group(1) if older else ""),
    }

    # Legacy artifact: when "Diagnoses (Older):" is blank, the original parser
    # ran on and captured the next section header as a diagnosis.
    if LEGACY_QUIRKS and not out["older"]:
        out["older"] = [SEC_MEDS]

    return out


def parse_medications(ehr: str) -> list[str]:
    block = _section(ehr, SEC_MEDS, None)
    meds: list[str] = []
    for line in block.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("-"):
            meds.append(line.lstrip("-").strip())
        elif LEGACY_QUIRKS and line.endswith(":"):
            # Legacy artifact: sub-headers such as
            # "Recently Ended Within 30-Days of Message:" were kept as items.
            meds.append(line)
    return meds


def parse_record(text: str) -> dict:
    """Parse one packed `text` blob into the structured record body."""
    if MSG_MARKER in text:
        ehr, message = text.split(MSG_MARKER, 1)
    else:
        ehr, message = text, ""

    return {
        "patient_message": message.strip(),
        "demographics": parse_demographics(ehr),
        "problem_list": parse_problem_list(ehr),
        "diagnoses": parse_diagnoses(ehr),
        "medications": parse_medications(ehr),
    }


# --------------------------------------------------------------------------
# Driver
# --------------------------------------------------------------------------

def infer_prefix(path: Path) -> str:
    """'INBOX_A-one_pt.parquet' -> 'INBOX_A'."""
    stem = path.stem
    return re.sub(r"[-_]one[-_]pt$", "", stem, flags=re.IGNORECASE)


def parquet_to_records(
    path: Path,
    prefix: str | None = None,
    keep_empty: bool = False,
    id_width: int = 2,
) -> list[dict]:
    """Convert one parquet file into a list of schema-conformant records."""
    df = pd.read_parquet(path)

    missing = {"text", "level", "relevancy"} - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing column(s) {sorted(missing)}")

    prefix = prefix or infer_prefix(path)
    records, skipped = [], 0

    for i, row in df.reset_index(drop=True).iterrows():
        text = row["text"]

        # Null / blank rows are padding in the export, not patients.
        if not isinstance(text, str) or not text.strip():
            skipped += 1
            if not keep_empty:
                continue
            rec = {
                "patient_message": "",
                "demographics": {"age": "", "gender": ""},
                "problem_list": [],
                "diagnoses": {"past_year": [], "older": []},
                "medications": [],
            }
        else:
            rec = parse_record(text)

        rec["level"] = _to_int(row["level"])
        rec["relevancy"] = _to_int(row["relevancy"])
        rec["id"] = f"{prefix}-{i:0{id_width}d}"
        records.append(rec)

    print(
        f"  {path.name}: {len(df)} rows -> {len(records)} records "
        f"({skipped} null/blank {'kept' if keep_empty else 'skipped'})",
        file=sys.stderr,
    )
    return records


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("parquet", nargs="+", type=Path, help="input .parquet file(s)")
    ap.add_argument("-o", "--out", type=Path, default=Path("pmr_synth_parsed.json"))
    ap.add_argument("--prefix", default=None,
                    help="ID prefix (default: inferred from filename)")
    ap.add_argument("--keep-empty", action="store_true",
                    help="emit placeholder records for null rows instead of skipping")
    ap.add_argument("--id-width", type=int, default=2,
                    help="zero-padding width for the numeric ID suffix (default: 2)")
    args = ap.parse_args()

    print(args)

    all_records: list[dict] = []
    for path in args.parquet:
        all_records.extend(
            parquet_to_records(path, args.prefix, args.keep_empty, args.id_width)
        )

    args.out.parent.mkdir(parents=True, exist_ok=True)
    with args.out.open("w", encoding="utf-8") as fh:
        json.dump(all_records, fh, indent=2, ensure_ascii=False)

    print(f"Wrote {len(all_records)} record(s) -> {args.out}", file=sys.stderr)


if __name__ == "__main__":
    main()


Overwriting parse_pmr_parquet.py


In [ ]:
|

In [ ]:
import pandas as pd

parquet_path = '/content/INBOX_A-00000-of-00001.parquet'

# Read the parquet file
df = pd.read_parquet(parquet_path)

# Display the first few rows
display(df.head())

,text,level,relevancy
0,### EHR: ###Demographics###\nAge: Between 75 -...,2,7
1,### EHR: ###Demographics###\nAge: Between 25 -...,1,10
2,### EHR: ###Demographics###\nAge: Between 25 -...,4,3
3,### EHR: ###Demographics###\nAge: Between 25 -...,3,5
4,### EHR: ###Demographics###\nAge: Between 45 -...,1,10


In [ ]:
import json

output_json_path = '/content/pmr_synth_parsed_all.json'

# Helper function to convert stringified JSON back to Python objects
def parse_json_string(val):
    if isinstance(val, str):
        val_stripped = val.strip()
        if (val_stripped.startswith('{') and val_stripped.endswith('}')) or \
           (val_stripped.startswith('[') and val_stripped.endswith(']')):
            try:
                return json.loads(val_stripped)
            except json.JSONDecodeError:
                return val
    return val

# Apply the parser to all columns to reconstruct the nested structure
for col in df.columns:
    df[col] = df[col].apply(parse_json_string)

# Convert DataFrame to a list of dictionaries (records)
patients_data = df.to_dict(orient='records')

# Save to JSON
with open(output_json_path, 'w') as f:
    json.dump(patients_data, f, indent=2)

print(f"Successfully processed {len(patients_data)} records and saved to {output_json_path}")

Successfully processed 30 records and saved to /content/pmr_synth_parsed_all.json


In [ ]:
1#{ "input_text": "Hypertension, unspecified type", "cui": "C0020538",
#  "sctid": "38341003", "snomed_name": "Hypertensive disorder" }

In [ ]:
client = UMLSClient(api_key=UMLS_API_KEY)

In [ ]:
result = resolve_to_snomed(client, "Hypertension, unspecified type")

In [ ]:
#History of cardiomyopathy
result = resolve_to_snomed(client, "History of cardiomyopathy")

In [ ]:
print(result)

{'input_text': 'History of cardiomyopathy', 'cui': 'C4039015', 'sctid': '690491000119104', 'snomed_name': 'History of cardiomyopathy'}


In [ ]:
# --------------------------------------------------------------------------- #
# Offline self-test with a mock ontology (no key / network needed)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    class MockOntology:
        # code -> its ancestors (self added automatically); mirrors is-a subsumption
        ANC = {
            "408512008": {"414916001"},   # Severe obesity  is-a  Obesity
            "238136002": {"414916001"},   # Morbid obesity  is-a  Obesity
        }
        def is_a(self, code, parent):
            return parent == code or parent in self.ANC.get(code, set())

    onto = MockOntology()
    cases = {
        "Severe obesity only [408512008]":               (["408512008"], 1),
        "ESRD only [46177005] (no CVD)":                  (["46177005"], 3),
        "CKD only [709044004]":                           (["709044004"], 2),
        "T2DM + AF [44054006,49436004] (overlap)":        (["44054006", "49436004"], 4),
        "AF only [49436004] (isolated CVD, overlap on)":  (["49436004"], 0),
        "AF only, overlap OFF":                           (["49436004"], 4),
        "nothing relevant [22222222]":                    (["22222222"], 0),
    }
    print("Corrected cascade self-test (mock ontology):\n")
    for label, (codes, expected) in cases.items():
        overlap = "OFF" not in label
        got = approx_ckm_stage(codes)
        flag = "OK" if got == expected else f"!! expected {expected}"
        print(f"  Stage {got}  {flag:14s} {label}")
    print("\n  Severe obesity -> 1 (not 0); ESRD-only -> 3 (not 4); CKD -> 2 (not 3);")
    print("  isolated AF -> 0 with overlap on, 4 with it off. All per Table 4.")
    print(onto)
## TODO: Get expected from string matched codes

Corrected cascade self-test (mock ontology):



TypeError: approx_ckm_stage() missing 1 required positional argument: 'snomed_ontology'